## 基本用法

LangChain内置工具列表：https://docs.langchain.com/oss/python/integrations/tools
Agents支持绑定一个或多个工具。

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

### 1、绑定一个工具

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from rich import print as rprint

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    查询天气工具

    Args:
        city: 城市名称
    """
    return f"{city}的天气很好"

agent = create_agent(
    model=model,
    tools=[get_weather]
)

resp = agent.invoke({
    "messages": [
        {"role": "system", "content": "你是一个天气查询助手，只回答天气相关的问题，其他问题请直接回答：我不清楚这问题答案。"},
        {"role": "user", "content": "北京的天气怎么样？"}
    ]
})

rprint(resp)
rprint(resp['messages'][-1].content)

### 2、接入内置工具

绑定内置的TavilySearch搜索工具，可以借助Tavily进行网络搜索和信息爬取。

这里我们需要先在tavily官网注册并获得API-KEY（每月有免费额度）：https://www.tavily.com/。

In [ ]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
from rich import print as rprint

load_dotenv(override=True)

web_search = TavilySearch(
    tavily_api_key=os.getenv("TAVILY_API_KEY"),
    max_results=2
)

agent = create_agent(
    model=model,
    tools=[web_search]
)

result = agent.invoke({
    "messages": [{
        "role": "user", "content": "我是中国人，姆巴佩补水啦哪里有得卖？"
    }]
})

rprint(result)
rprint(result['messages'][-1].content)

### 3、绑定多个工具

In [ ]:

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    return f"{city}今天天气挺好"

@tool(parse_docstring=True)
def get_news():
    """
    新闻查询工具
    """
    return "近期，受全球储蓄芯片短缺等多重因素影响，多地回收商称废旧手机回收市场迎来“火热潮”，回收价格普遍上涨，旧手机成“香饽饽”。"

agent = create_agent(
    model=model,
    tools=[get_weather, get_news]
)

response = agent.invoke({
    "messages": ["你好，杭州今天的天气如何？今天有哪些新闻？"]
})
rprint(response)